# IEI097 · Laboratorio 2.12 — MLP + MLflow (cáncer de mama)

**Nombre:** Miguel Chura  
**N.º de lista:** `7` (es mi semilla `SEED`, definida en la primera celda de código del Paso 1)  
**Fecha:** 2026-11-08  
**Dataset:** Breast Cancer Wisconsin (Diagnostic) · `uciml/breast-cancer-wisconsin-data`

## Paso 1 · Entorno y semilla personal

In [ ]:
!pip install -q mlflow pyngrok kagglehub

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, mlflow

# ---- TU NÚMERO DE LISTA = tu semilla personal ----
SEED = 0          # <-- CAMBIA 0 por tu número de lista
assert SEED != 0, "Define SEED con tu número de lista antes de continuar"
random.seed(SEED); np.random.seed(SEED)

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("mlflow:", mlflow.__version__)
print("SEED:", SEED)

## Paso 2 · Carga y preparación de los datos

In [ ]:
import kagglehub
from google.colab import userdata

# Credenciales desde Colab Secrets (como en el Laboratorio 02)
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

ruta = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print("Archivos:", os.listdir(ruta))

In [ ]:
csv = [f for f in os.listdir(ruta) if f.endswith(".csv")][0]   # confirma que es el CSV principal
df = pd.read_csv(os.path.join(ruta, csv))
print(csv, df.shape)
df.info()

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0])   # columna totalmente vacía (Unnamed: 32)

df = df.drop(columns=["Unnamed: 32", "id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})   # maligno = 1 (clase de interés)

print(df["diagnosis"].value_counts())
print(df["diagnosis"].value_counts(normalize=True).round(3))

**Balance de clases:** hay **357 benignos (B, 62,7 %)** y **212 malignos (M, 37,3 %)**, 569 pacientes en total. No es un desbalance extremo, pero la clase maligna (la de interés) es minoritaria: una exactitud alta puede esconder malignos mal clasificados, por eso además de `acc_test` reporto `recall_maligno` y `f1_maligno`.

In [ ]:
cols = ["radius_mean", "area_mean", "smoothness_mean"]
rangos = df[cols].agg(["min", "max"]).T
rangos["rango"] = rangos["max"] - rangos["min"]
print(rangos)
print("area_mean / smoothness_mean (veces mayor el rango):",
      round(rangos.loc["area_mean", "rango"] / rangos.loc["smoothness_mean", "rango"], 1))

**Evidencia de escala:** el rango de `area_mean` (≈ 2.357) es unas **21.283 veces mayor** que el de `smoothness_mean` (≈ 0,11). Para un MLP esto es un problema: las variables de magnitud grande dominan la suma ponderada de la primera capa y el gradiente, saturan las activaciones y hacen que el descenso del gradiente avance lento o se estanque. Por eso conviene estandarizar (`StandardScaler`, ajustado solo con el train para evitar fuga de información).

## Paso 3 · Partición (fija durante todo el laboratorio)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="diagnosis")
y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print("Proporción maligno · total:", round(y.mean(), 3),
      "· train:", round(y_train.mean(), 3), "· test:", round(y_test.mean(), 3))
print(X_train.shape, X_test.shape)

## Paso 4 · MLflow en Colab (guarda `NGROK_AUTHTOKEN` en Secrets y activa el acceso desde el notebook)

In [ ]:
import subprocess, requests
from google.colab import userdata
from pyngrok import ngrok

# Servidor de seguimiento con SQLite como backend store.
# MLflow >= 3.5 rechaza Host distintos de localhost; estas variables permiten abrir la UI por ngrok.
env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS="*",
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")
subprocess.Popen(["mlflow", "server",
                  "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"], env=env)

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()                    # cierra túneles anteriores
tunel = ngrok.connect(5000)
print("MLflow UI:", tunel.public_url)

> Abre la URL, pulsa **Visit Site** si ngrok muestra un aviso y comprueba que ves la interfaz de MLflow. No ejecutes la celda anterior dos veces seguidas.

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Chura")

## Paso 5 · Experimentos controlados (un cambio a la vez)

**Hipótesis A vs. B:**  
Creo que el escalado **mejorará** el resultado. Sin escalar, variables como `area_mean` (rango ≈ 21.000 veces el de `smoothness_mean`) dominan el cálculo de la primera capa y el gradiente; las neuronas se saturan y el descenso del gradiente optimiza mal las variables de escala pequeña. Con `StandardScaler` todas quedan con media 0 y desviación 1, el entrenamiento es más estable y debería subir la exactitud en test y, sobre todo, el recall de la clase maligna.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay)
from mlflow.entities import Metric
from mlflow import MlflowClient

client = MlflowClient()

def correr_experimento(nombre, escalar, capas, activacion, fase,
                       seed_modelo=SEED, alpha=1e-4, max_iter=500):
    pasos = [("scaler", StandardScaler())] if escalar else []
    pasos.append(("mlp", MLPClassifier(hidden_layer_sizes=capas, activation=activacion,
                                       alpha=alpha, max_iter=max_iter,
                                       random_state=seed_modelo)))
    pipe = Pipeline(pasos)          # el escalado se ajusta solo con train (sin fuga de datos)
    pipe.fit(X_train, y_train)
    mlp = pipe.named_steps["mlp"]

    pred_tr, pred_te = pipe.predict(X_train), pipe.predict(X_test)
    m = {
        "acc_train": accuracy_score(y_train, pred_tr),
        "acc_test": accuracy_score(y_test, pred_te),
        "recall_maligno": recall_score(y_test, pred_te, pos_label=1),
        "f1_maligno": f1_score(y_test, pred_te, pos_label=1),
        "n_iter": float(mlp.n_iter_),
    }
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.set_tag("fase", fase)
        mlflow.log_params({"arquitectura": str(capas), "activacion": activacion,
                           "alpha": alpha, "max_iter": max_iter,
                           "random_state": seed_modelo, "escalado": escalar,
                           "seed_particion": SEED})
        mlflow.log_metrics(m)
        ahora = int(time.time() * 1000)
        client.log_batch(run.info.run_id, metrics=[
            Metric("train_loss", float(v), ahora, i) for i, v in enumerate(mlp.loss_curve_)])
        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, pred_te, ax=ax,
            display_labels=["Benigno", "Maligno"], cmap="Blues")
        ax.set_title(nombre)
        mlflow.log_figure(fig, "matriz_confusion.png")
        plt.close(fig)
    print(f"{nombre}: " + " · ".join(f"{k}={v:.4f}" for k, v in m.items()))
    return m

resultados = {}

In [ ]:
resultados["A"] = correr_experimento("A_sin_escalar_30_relu", False, (30,), "relu", "experimentos")

**Interpretación A:** punto de partida (MLP de 30 neuronas, ReLU, sin escalar). Es el peor modelo: `acc_test` = 0,921, `recall_maligno` = 0,833 (detecta 35 de 42 malignos; 7 pasan como benignos) y `f1_maligno` = 0,886. Su `train_loss` final es la más alta (0,153) y `acc_train` solo 0,941, o sea que ni siquiera ajustó bien los datos de entrenamiento: la optimización quedó frenada por las escalas distintas.

In [ ]:
resultados["B"] = correr_experimento("B_escalado_30_relu", True, (30,), "relu", "experimentos")

**Interpretación B:** mi hipótesis **se cumplió**. Al escalar (todo lo demás igual que A) `acc_test` sube de 0,921 a 0,974, `recall_maligno` de 0,833 a 0,952 (ahora detecta 40 de 42 malignos) y `train_loss` baja de 0,153 a 0,018. Además `n_iter` sube de 225 a 387: con datos escalados el optimizador sigue mejorando más iteraciones en vez de estancarse. Un solo cambio (escalar) explica una mejora de 6 pacientes bien clasificados en test.

In [ ]:
resultados["C"] = correr_experimento("C_escalado_64-32_relu", True, (64, 32), "relu", "experimentos")

**Interpretación C:** pasar de (30) a (64, 32) neuronas **no mejoró** el resultado de test: `acc_test` 0,965 y `recall_maligno` 0,929 (con seed 7), apenas por debajo de B (0,974 y 0,952). Sí cambió el ajuste de train: `acc_train` = 1,0 y `train_loss` = 0,004, con convergencia más rápida (`n_iter` 155 vs 387). Es decir, el modelo más grande memoriza mejor el train pero no generaliza mejor; la diferencia con B es de 1 paciente, dentro de la variación entre semillas.

In [ ]:
resultados["D"] = correr_experimento("D_escalado_64-32_tanh", True, (64, 32), "tanh", "experimentos")

**Interpretación D:** cambiar ReLU por `tanh` (misma arquitectura (64, 32) que C) dio `acc_test` 0,974 y `recall_maligno` 0,952, igual que B y 1 paciente mejor que C. Pero `tanh` necesitó más iteraciones (282 vs 155) y terminó con `train_loss` = 0,0055 (un poco más alta que C). Como las semillas 8 a 11 de C también llegan a 0,974 / 0,952, no puedo atribuir esa mejora a la activación: es ruido de semilla. Conclusión: la activación no tuvo un efecto claro en este dataset.

In [ ]:
pd.DataFrame(resultados).T.round(4)

**Opcional:** agrega 1 o 2 runs variando `alpha` o `learning_rate_init` (añade el parámetro a `correr_experimento` si lo usas).

## Paso 6 · Semilla y replicación

In [ ]:
semillas = {}
for k in range(5):
    s = SEED + k
    semillas[s] = correr_experimento(f"C_seed{s}", True, (64, 32), "relu", "semillas", seed_modelo=s)

tabla = pd.DataFrame(semillas).T
resumen = pd.DataFrame({
    "min": tabla[["acc_test", "recall_maligno"]].min(),
    "max": tabla[["acc_test", "recall_maligno"]].max()})
resumen["rango"] = resumen["max"] - resumen["min"]
resumen.round(4)

In [ ]:
replica = correr_experimento(f"C_replica_seed{SEED}", True, (64, 32), "relu", "replica")

comp = pd.DataFrame({"C (original)": resultados["C"], "réplica": replica})
comp["diferencia"] = (comp["C (original)"] - comp["réplica"]).abs()
display(comp)
print("¿Idénticos?", bool((comp["diferencia"] < 1e-12).all()))

**Replicación:** **sí coinciden.** La réplica (`C_replica_seed7`) dio exactamente las mismas métricas que el run C original (`acc_test` 0,9649, `recall_maligno` 0,9286, `n_iter` 155, `train_loss` 0,0043); la diferencia es 0 (< 1e-12). Esto ocurre porque fijé las dos fuentes de aleatoriedad: la partición train/test (`seed_particion` = 7) y la inicialización/barajado del MLP (`random_state` = 7). Además coincide con el run `C_seed7` de la serie de semillas.

## Paso 7 · Evidencia

En la interfaz de MLflow (con la URL de ngrok visible en la barra de direcciones) toma **3 capturas** y guárdalas en `evidencias/`:

1. Tabla de ejecuciones ordenada por `recall_maligno` (≥ 10 runs; se ven las dos con la misma semilla).
2. **Compare** de al menos 4 runs, o gráfico de `train_loss` con ≥ 2 runs superpuestos.
3. Una ejecución abierta con el artefacto `matriz_confusion.png`.

In [ ]:
runs = mlflow.search_runs(experiment_names=["IEI097-L2.12-Chura"])
cols = [c for c in runs.columns if c.startswith(("params.", "metrics.", "tags.fase"))] + ["tags.mlflow.runName"]
runs_out = runs[cols].sort_values("metrics.recall_maligno", ascending=False)
runs_out.to_csv("runs.csv", index=False)
print(len(runs_out), "runs exportadas")
runs_out.head(12)

In [ ]:
from google.colab import files
files.download("runs.csv")

## Paso 8 · Preguntas de cierre

**1. ¿Qué cambió al escalar las variables?**  
Con todo lo demás igual, escalar mejoró fuertemente el modelo: del run A (sin escalar) al B (escalado) `acc_test` pasó de 0,921 a 0,974, `recall_maligno` de 0,833 a 0,952 y `train_loss` de 0,153 a 0,018. Concepto: el _feature scaling_ (estandarizar a media 0 y desviación 1) pone todas las variables en la misma escala. Sin él, `area_mean` tiene un rango unas 21.000 veces mayor que `smoothness_mean`, domina las sumas ponderadas y el gradiente, y el descenso del gradiente optimiza mal. Es la mayor mejora de todo el laboratorio (+5,3 puntos de exactitud y +11,9 de recall), mucho mayor que cualquier otro cambio.

**2. ¿Más capas y neuronas mejoraron el resultado? ¿Cómo lo sé, considerando la variación entre semillas?**  
No. B (30 neuronas) obtuvo `acc_test` 0,974 y C (64, 32) obtuvo 0,965 con seed 7. Pero al repetir C con 5 semillas (7 a 11) su `acc_test` fue 0,965 en una semilla y 0,974 en las otras cuatro: el valor de B cae dentro del rango de C. Por lo tanto la diferencia entre B y C es ruido de semilla y no hay evidencia de que la arquitectura mayor ayude; solo hizo `acc_train` = 1,0 (memoriza el train) y convergió más rápido.

**3. Con mis 5 semillas, ¿qué tan grande fue la variación? ¿Es mayor o menor que las diferencias entre configuraciones? ¿Qué implica?**  
Con 5 semillas (7 a 11) de la configuración C: `acc_test` varió entre 0,9649 y 0,9737 (**rango 0,0088**) y `recall_maligno` entre 0,9286 y 0,9524 (**rango 0,0238**). En el test de 114 pacientes (72 benignos y 42 malignos) eso equivale a 1 paciente de diferencia. Esa variación es **mucho menor** que el efecto de escalar (0,053 de exactitud y 0,119 de recall), pero **del mismo tamaño** que las diferencias entre B, C y D (1 paciente). Implicación: solo puedo afirmar que una configuración es “mejor” si su diferencia supera claramente la variación entre semillas (como pasa con escalar); para diferencias de 1 paciente necesito repetir con varias semillas o usar validación cruzada y reportar media ± desviación, no un único run.

**4. En diagnóstico de cáncer, ¿qué error es más costoso: falso negativo o falso positivo? ¿Qué métrica priorizaría?**  
El **falso negativo** (un tumor maligno clasificado como benigno): el paciente no recibe tratamiento a tiempo y el costo puede ser la vida. Un falso positivo genera exámenes adicionales y angustia, pero se corrige con una biopsia. Por eso priorizaría el **recall (sensibilidad) de la clase maligna**, vigilando que el `f1_maligno` o la precisión no se desplomen. Ejemplo en mis runs: el modelo B (matriz de confusión: 71 verdaderos negativos, 1 falso positivo, 2 falsos negativos, 40 verdaderos positivos) deja 2 malignos sin detectar; el A dejaba 7.

**5. ¿Qué información de MLflow no tendría con solo `print()`? ¿Qué más registraría para que otra persona replique mi experimento?**  
Con `print()` los resultados se pierden al cerrar el notebook y no se pueden comparar. MLflow guarda de forma persistente y ordenable cada run con sus parámetros, métricas, la curva de `train_loss`, artefactos (la matriz de confusión) y etiquetas (`fase`), y permite comparar runs lado a lado (Compare), filtrar y exportar (`runs.csv`). Para que otra persona replique mi experimento registraría además: versiones de Python, scikit-learn y MLflow, la versión/fuente del dataset (Kaggle `uciml/breast-cancer-wisconsin-data`, 569 filas), el tamaño y la semilla de la partición, el preprocesamiento (`StandardScaler`), el código o commit usado y el tiempo de ejecución.

In [ ]:
ngrok.kill()   # cierra el túnel (el servidor MLflow queda expuesto mientras esté abierto)
print("Túnel cerrado. Exporta runs.csv y sube el notebook + evidencias/ a GitHub antes de cerrar Colab.")